# Creation of STAC collection for ICESat-2 ATL08 version 6

This script describes the creation of ICESat-2_ATL08v6 stac collection, which is published on OpenLandMap STAC (https://stac.openlandmap.org/ICESat-2_ATL08v6/collection.json). 

In [ ]:
import os
import json
import rasterio
import urllib.request
import pystac
from minio import Minio
from datetime import datetime, timezone, timedelta
from shapely.geometry import Polygon, mapping
from tempfile import TemporaryDirectory
from shapely import from_wkb
import duckdb
import contextily as cx
from shapely import geometry
import geopandas as gpd
import rasterio
import numpy as np
from rasterio.transform import from_bounds
from rasterio.features import rasterize
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from minio import Minio
import struct
from shapely.geometry import Point
import geopandas as gpd
import math

## Part 1: Retrieve files from s3 buckets

In [ ]:
access_key=''
secret_access_key=''
s3_ip='192.168.49.30:8333'

s3_config = {
'access_key': access_key,
'secret_access_key': secret_access_key,
'host': s3_ip}
client = Minio(s3_config['host'], s3_config['access_key'], s3_config['secret_access_key'], secure=False) 
olm_gedi_path="atl08v6.icesat_20181014_20230621_go_epsg.4326_v20250315"
collection_name="ICESat-2_ATL08v6"

In [ ]:
# List all objects in the bucket
objects=client.list_objects("global", recursive=True, prefix=f"glidar/icesat-ard/atl08v006/{olm_gedi_path}")
# Print file names
files =[]
for obj in [i for i in objects if i.object_name.endswith('.parquet')]:    
    files.append([obj.object_name,obj.size])

## Part 2: define fucntions

1. transfer geoparquet wbk to wkt
2. rasterize goepandas dataframe to geotiff

In [ ]:
def transfer(geom_byte):
    # Given bytearray
    byte_data = geom_byte

    # Extract X and Y coordinates (assuming IEEE 754 double precision format)
    x, y = struct.unpack("dd", byte_data[-16:])  # Extract last 16 bytes as two doubles

    # Create a Shapely Point
    point = Point(x, y)
    return point


In [ ]:
def rasterize_gdf_to_geotiff(gdf, column, output_tiff, resolution=10, nodata_value=0):
    """
    Rasterizes a GeoDataFrame using values from a specific column and saves it as a GeoTIFF.
    
    Parameters:
    - gdf: GeoDataFrame containing geometries.
    - column: Column name whose values will be burned into the raster.
    - output_tiff: Output path for the GeoTIFF file.
    - resolution: Pixel resolution in the units of the CRS.
    """
    # Get bounds of the vector data
    minx, miny, maxx, maxy = gdf.buffer(resolution).total_bounds

    # Define raster width and height
    width = int((maxx - minx) / resolution)
    height = int((maxy - miny) / resolution)

    # Define transformation (maps pixel coordinates to spatial coordinates)
    transform = from_bounds(minx, miny, maxx, maxy, width, height)

    # Create a list of (geometry, value) tuples
    shapes = [(geom, value) for geom, value in zip(gdf.geometry, gdf[column])]

    # Rasterize vector data into an array
    raster = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=nodata_value,  # Background value
        all_touched=True,  # If True, touches any pixel it overlaps
    )
    
    # Normalize raster values to range [0, 1] for colormap
    min_val, max_val = raster[raster > nodata_value].min(), raster.max()
    normalized_raster = (raster - min_val) / (max_val - min_val)
    normalized_raster[raster == nodata_value] = 0  # Keep NoData as 0

    # Apply "magma" colormap from Matplotlib
    cmap = plt.cm.magma
    rgba_img = cmap(normalized_raster)  # Returns an (H, W, 4) array with RGBA values

    # Convert RGBA to 3-band (RGB) array (scale to 0–255)
    rgb_raster = (rgba_img[:, :, :3] * 255).astype(np.uint8)

    # Save to Cloud-Optimized GeoTIFF (COG)
    with rasterio.open(
        output_tiff, "w",
        driver="COG",  # Saves as a Cloud-Optimized GeoTIFF
        height=height, width=width,
        count=1, dtype=rasterio.uint8,  # 3 bands (RGB)
        crs=gdf.crs, transform=transform,
        nodata=0,  # Set NoData value
        compress="DEFLATE",  # Apply compression
        tiled=True,  # Enable tiling for COG
        blockxsize=256, blockysize=256,  # Set block size for COG
        BIGTIFF="IF_NEEDED"  # Allow large files
    ) as dst:
        dst.write(raster, 1)  # one band
        
        #dst.write(rgb_raster[:, :, 0], 1)  # Red band
        #dst.write(rgb_raster[:, :, 1], 2)  # Green band
        #dst.write(rgb_raster[:, :, 2], 3)  # Blue band
    
    print(f"Raster saved to {output_tiff}")

## Part 3: Process the files into STAC items and create metadata

STAC items of GEDI02 collection contains the infos: (1) url points to the single geoparquet file in S3 server, (2) an overview image of rasterized GEDI points as COG as thumbnail, (3) auxiliary metadata, such as footprint, bbox, platform, license.

In [ ]:
def worker(file):
    url='https://s3.opengeohub.org/global/'+file[0]
    asset_name = file[0].split('/')[3]
    file_size = file[1]

    df_duckdb = duckdb.sql(f"""
                            INSTALL httpfs;
                            LOAD httpfs;
                            INSTALL spatial;
                            LOAD spatial;

                            SELECT latitude_20m, longitude_20m, med_ht, start_dt, end_dt, geometry

                            FROM "{url}"
                            """)

    df=df_duckdb.df()
    try:
        df['geometry']=df.geometry.apply(lambda x: transfer(x))
        num_of_points=len(df)

        s_date=min(df.start_dt)
        e_date=max(df.end_dt)

        item_name = '_'.join(file[0].split('/')[4:]).split('.')[0]


        gdf = gpd.GeoDataFrame(
            df, geometry=gpd.points_from_xy(df.longitude_20m, df.latitude_20m), crs="EPSG:4326"
        )
        bbox=gdf.total_bounds.tolist()
        xmin=math.floor(bbox[0])
        ymin=math.ceil(bbox[1])
        xmax=math.floor(bbox[2])
        ymax=math.ceil(bbox[3])

        footprint = mapping(geometry.box(xmin,ymin,xmax,ymax))
        if len(gdf)>5000:
            gdf = gdf.sample(5000)
        os.makedirs(f'stac/{collection_name}/{item_name}',exist_ok=True)
        thumbmail_path = f'stac/{collection_name}/{item_name}/overview_{item_name}.tif'
        rasterize_gdf_to_geotiff(gdf,'med_ht',thumbmail_path,0.01, nodata_value=0)

        item = pystac.Item(id=item_name,
                         geometry=footprint,
                         bbox=gdf.total_bounds.tolist(),
                         properties={'size (bytes)':file_size,'point counts':num_of_points},
                         start_datetime=s_date,
                         end_datetime=e_date,
                         datetime=None,
                         stac_extensions='https://github.com/Open-Earth-Monitor/GlobalEarthPoint',                   
                         )

        item.common_metadata.platform = 'GlobalEarthPoint'
        item.common_metadata.license = 'CC-BY-4.0'
        # Define the COG asset with media type
        item.assets[asset_name]=pystac.Asset(
            href=url,
            roles=["data"],
            title="GeoParquet",
            description='A partition of photon-based ICESat-2 ATL08v6 dataset in GeoParquet.'
        )
        #item.add_asset(
        #    key=asset_name,
        #    title='GeoParquet',
        #    roles=["data"],        
        #    asset=pystac.Asset(
        #        href=url,
        #        description='A partition of photon-based ICESat-2 ATL08v6 dataset in GeoParquet.'
        #    )

        #)
        item.assets["thumbnail"] = pystac.Asset(
            href=f'overview_{item_name}.tif',
            media_type="image/tiff; application=geotiff; profile=cloud-optimized",
            roles=["overview","thumbnail"],
            title="Overview",
            description="A COG representing median height rasterized from ICESat-2 photons in a segment (vector data), 1km."
        )
        # Add Asset and all its information to Item 

        return item
    except:
        return file

In [ ]:
items = Parallel(n_jobs=30)(delayed(worker)(i) for i in files)

In [ ]:
can_items=[]
for i in items:
    if not isinstance(i, list):
        can_items.append(i)
        

In [ ]:
no_can_items=[]
for file in items:
    if isinstance(file, list):
        url='https://s3.opengeohub.org/global/'+file[0]
        asset_name = file[0].split('/')[3]
        file_size = file[1]

        df_duckdb = duckdb.sql(f"""
                                INSTALL httpfs;
                                LOAD httpfs;
                                INSTALL spatial;
                                LOAD spatial;

                                SELECT latitude_20m, longitude_20m, h_te_best_fit_20m ,med_ht, start_dt, end_dt, geometry

                                FROM "{url}"
                                """)

        df=df_duckdb.df()
        df['geometry']=df.geometry.apply(lambda x: transfer(x))
        num_of_points=len(df)

        s_date=min(df.start_dt)
        e_date=max(df.end_dt)

        item_name = '_'.join(file[0].split('/')[4:]).split('.')[0]


        gdf = gpd.GeoDataFrame(
            df, geometry=gpd.points_from_xy(df.longitude_20m, df.latitude_20m), crs="EPSG:4326"
        )
        bbox=gdf.total_bounds.tolist()
        xmin=math.floor(bbox[0])
        ymin=math.ceil(bbox[1])
        xmax=math.floor(bbox[2])
        ymax=math.ceil(bbox[3])

        footprint = mapping(geometry.box(xmin,ymin,xmax,ymax))
        if len(gdf)>5000:
            gdf = gdf.sample(5000)
        os.makedirs(f'stac/{collection_name}/{item_name}',exist_ok=True)
        thumbmail_path = f'stac/{collection_name}/{item_name}/overview_{item_name}.tif'
        try:
            rasterize_gdf_to_geotiff(gdf,'med_ht',thumbmail_path,0.01, nodata_value=0)
            overview=True
        except:
            overview=False
        item = pystac.Item(id=item_name,
                         geometry=footprint,
                         bbox=gdf.total_bounds.tolist(),
                         properties={'size (bytes)':file_size,'point counts':num_of_points},
                         start_datetime=s_date,
                         end_datetime=e_date,
                         datetime=None,
                         stac_extensions='https://github.com/Open-Earth-Monitor/GlobalEarthPoint',                   
                         )

        item.common_metadata.platform = 'GlobalEarthPoint'
        item.common_metadata.license = 'CC-BY-4.0'
        # Define the COG asset with media type
        item.assets[asset_name]=pystac.Asset(
            href=url,
            roles=["data"],
            title="GeoParquet",
            description='A partition of photon-based ICESat-2 ATL08v6 dataset in GeoParquet.'
        )
        #item.add_asset(
        #    key=asset_name,
        #    title='GeoParquet',
        #    roles=["data"],        
        #    asset=pystac.Asset(
        #        href=url,
        #        description='A partition of photon-based ICESat-2 ATL08v6 dataset in GeoParquet.'
        #    )

        #)
        if overview:
            item.assets["thumbnail"] = pystac.Asset(
                href=f'overview_{item_name}.tif',
                media_type="image/tiff; application=geotiff; profile=cloud-optimized",
                roles=["overview","thumbnail"],
                title="Overview",
                description="A COG representing median height rasterized from ICESat-2 photons in a segment (vector data), 1km."
            )
        no_can_items.append(item)

In [ ]:
final_items= can_items + no_can_items

# Part 4: create a geojson of tile system overview at the collection level.
The GeoJSON overview contains the tiling system (5x5 degree) yearly, the size, point counts, startdate and enddate of the record, etc. The purpose of this file is inspired by [stac-geoparquet](https://github.com/stac-utils/stac-geoparquet), where the overview of collection can help filter the collection without loading the items themselves. 

In [ ]:
df=[]
for item in final_items:
    d={'item_id':item.id,
'size_in_mb':item.properties['size (bytes)']/(1024**2),
'point_counts':item.properties['point counts'],
'start_date':item.properties['start_datetime'],
'end_date':item.properties['end_datetime'],
'platform':item.properties['platform'],
'stac_extentsion':item.stac_extensions,
'asset_file':item.assets[olm_gedi_path].href,
'geometry':Polygon(item.geometry['coordinates'][0])}
    df.append(d)


In [ ]:
df = pd.DataFrame(df)

In [ ]:
gdf = gpd.GeoDataFrame(
    df, crs="EPSG:4326"
)

In [ ]:
gdf=gdf.set_geometry('geometry')

In [ ]:
gdf.to_file(f'stac/{collection_name}/stac_items.geojson', driver='GeoJSON')

0## Part 5: Create a collection placeholder 

Create a collection and define the spatial and temporal extent, license, columns, keywords, etc..

In [ ]:
collection_bbox = [-180, -88, 180 ,84]
collection_interval = [datetime(2018,10,14), datetime(2023,6,21)]

In [ ]:
spatial_extent = pystac.SpatialExtent(bboxes=[collection_bbox])
temporal_extent = pystac.TemporalExtent(intervals=[collection_interval])

In [ ]:
collection_extent = pystac.Extent(spatial=spatial_extent, temporal=temporal_extent)

In [ ]:
collection = pystac.Collection(id=collection_name,
                               title='OpenLandMap ICESat-2 ATL08 version 6',
                               description="ICESat-2 (Ice, Cloud, and land Elevation Satellite 2) is the Advanced Topographic Laser Altimeter System (ATLAS), a space-based lidar. The derived product ATL08 (version 6) contains along-track heights above the WGS84 ellipsoid (ITRF2014 reference frame) for the ground and canopy surfaces. In our product, we append the individual photon heights in 20m segment initially stored in ATL03 products." ,
                               extent=collection_extent,
                               license='CC-BY-4.0',
                               keywords=['ICESat-2','ATL08','version 6','in-situ data','lidar','canopy height','canopy structure','terrain height'],
                               extra_fields={'columns_provided':vector_columns})

In [ ]:
# Define a new provider
new_provider = pystac.Provider(
    name="OpenGeoHub",
    description="OpenGeoHub Foundation",
    roles=["processor", "host"],  # Can be 'producer', 'processor', 'host', 'licensor'
    url="http://opengeohub.org"  # Provider's website
)

# Add the provider to the collection (or update existing ones)
collection.providers = [new_provider]

In [ ]:
# Add DOI, Contact Name, and Email to extra_fields
collection.extra_fields["doi"] = "https://doi.org/10.5281/zenodo.8406375"  # Replace with actual DOI
collection.extra_fields["contact_name"] = "Yu Feng HO"
collection.extra_fields["contact_email"] = "yu-feng.ho@opengeohub.org"


In [ ]:
def make_serializable(d):
    for k, v in list(d.items()):
        if isinstance(v, (np.integer, np.floating)):
            d[k] = v.item()
        elif isinstance(v, (np.ndarray, pd.Index)):
            d[k] = v.tolist()
        elif isinstance(v, dict):
            make_serializable(v)

# Clean extra_fields if needed
make_serializable(collection.extra_fields)

## Part 6: Insert objects into Collection

In [ ]:
# insert overview of the collection (GeoJSON) 
tiles_vector = pystac.Asset(title='GeoJSON STAC items',href='stac_items.geojson', 
                          media_type=pystac.MediaType.GEOJSON,
                          description="STAC items of ICESat-2 ATL08 version 6 collection based on a 5 degree x 5 degree tiling system")
collection.add_asset(key='stac_items',
                   asset=tiles_vector)

In [ ]:
item

In [ ]:
item_path

In [ ]:
# insert items 
collection.add_items(items)

## Part 7: Create a self-contained catalog to host the collection

In [ ]:
catalog = pystac.Catalog(id='GlobalEarthPoint',
                         description='This Catalog serves the cloud optimized high equality vector data of the world')
catalog.add_child(collection)

In [ ]:
#catalog.normalize_hrefs(os.path.join('/mnt/apollo/eu_ecudatacube_vector', "stac"))
catalog.normalize_hrefs("stac")

In [ ]:
catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED,dest_href='stac')